# 03 - Your own datasets

Delphos represents each choice dataset using:

- a CSV data file
- a `dataset.yaml` schema
- attributes mapped to each alternative
- covariates and their discrete levels

This notebook shows how to inspect the datasets included with Delphos and how to structure your own dataset for use with the package.

### 0. Import Delphos

In [ ]:
import delphos as dp

### 1. Load a dataset
In this example, we use Swissmetro as an unseen dataset. 

In [ ]:
dataset = dp.load_dataset(4)
print(dataset)

### 2. Inspect dataset


You can see its information using alternatives, attributes, covariates from the dataset object. 

In [ ]:
print(f"Alternatives:   {dataset.alternatives}")
print(f"Attributes:     {dataset.attributes}")
print(f"Covariates:     {dataset.covariates}")

You can inspect both the dataset.yaml schema and the underlying CSV data.

In [ ]:
import yaml
import pandas as pd

# The YAML file defines how Delphos interprets the dataset:
schema_path = dataset.yaml_path
schema = yaml.safe_load(schema_path.read_text())

print("-Schema-")
for key in schema.keys():
    print(f"{key} - {schema[key]}")

# You can then load and inspect the dataset itself: 
df = pd.read_csv(dataset.dataset_path)
df


You can inspect the modelling terms that define the modelling space for the selected dataset.

In [ ]:
print("Alternatives")
for alt in dataset.alternatives:
    print(alt)

print("\nAttributes")
for attr in dataset.attributes:
    print(attr)

print("\nModelled covariates")
for cov in dataset.modelling_covariates:
    print(cov)


### 4. Catalogue and modelling space

Delphos uses a predefined catalogue to construct the model specification space.

The catalogue defines the identifiers associated with attributes, transformations, taste structures, and covariates:

<table>
<tr>
<td valign="top">

| ID | Attribute | Description |
|---:|---|---|
| 1 | ASC | Alternative-specific constant |
| 2 | IVTT | In-vehicle travel time |
| 3 | TC | Travel cost |
| 4 | OVTT | Out-of-vehicle travel time |
| 5 | TRA | Transfers |
| 6 | SQ | Service quality |
| 7 | Reliability | Reliability of the transport mode |

</td>
<td valign="top">

| ID | Covariate | Description |
|---:|---|---|
| 1 | Gender | Self-identified gender of the traveller |
| 2 | Income | Individual income; otherwise, household income |
| 3 | Age | Age of the individual |
| 4 | Purpose | Trip purpose |
| 5 | Car access | Car availability or access for the trip |
| 6 | Business | Indicator for a business trip |
| 7 | Education | Education level; otherwise, occupation |

</td>
</tr>
</table>


User-provided datasets must map their variables to the identifiers recognised by the trained Delphos agent. 

### 5. Create a user dataset folder

The example below uses the Swissmetro CSV as if it were user data. In your own work, replace `source_csv` and the schema dictionaries.


In [ ]:
from pathlib import Path
import delphos as dp

source_csv = dataset.dataset_path
user_folder = Path("tutorials/my_swissmetro_dataset")

id = 'id'

choice = 'choice'

panel = True

alternatives = {'TRAIN':    {'id': 1, 'avail': 'train_av'}, 
                'SM':       {'id': 2, 'avail': 'sm_av'}, 
                'CAR':      {'id': 3, 'avail': 'car_av'}}

attributes = {'time':    {'id': 2, 'mapping': {'TRAIN': 'train_tt_scaled', 'SM': 'sm_tt_scaled', 'CAR': 'car_tt_scaled'}},
              'cost':    {'id': 3, 'mapping': {'TRAIN': 'train_cost_scaled', 'SM': 'sm_cost_scaled', 'CAR': 'car_co_scaled'}},
              'headway': {'id': 4, 'mapping': {'TRAIN': 'train_he_scaled', 'SM': 'sm_he_scaled'}}, 'seat': {'id': 6, 'mapping': {'SM': 'sm_seats_scaled'}}}

covariates = {'purpose': {'id': 4, 'source': 'purpose', 'type': 'categorical', 'levels': [1, 2]},
              'first':   {'id': 6, 'source': 'first', 'type': 'categorical', 'levels': [0, 1]},
              'ticket':  {'id': None, 'source': 'ticket', 'type': 'categorical', 'levels': [1, 2, 3, 4, 5, 6, 7, 8, 10]},

              # other ..., 

              'ga':      {'id': None, 'source': 'ga', 'type': 'categorical', 'levels': [0, 1]}}

# Uncomment to create the folder when you are ready.
# user_task = dp.create_dataset(
#     user_folder,
#     name="MySwissmetro",
#     csv_path=source_csv,
#     choice_column=choice,
#     id_column=id,
#     panel=panel,
#     alternatives=alternatives,
#     attributes=attributes,
#     covariates=covariates,
#     ll_null=None,
#     ll_linear=None,
#     n_obs=None,
#     dataset_id=100,
# )
# print(user_task)
